# 🚀 ColabMCP - 稳定版服务器

## 优化内容
- ✅ 添加心跳保活机制，防止 Colab 休眠
- ✅ 自动重连 ngrok
- ✅ 更好的错误处理
- ✅ 防止服务意外停止

---
**获取 ngrok token**: https://dashboard.ngrok.com/get-started/your-authtoken

## 1️⃣ 安装依赖

In [ ]:
!pip install flask pyngrok requests psutil -q
print("✅ 依赖安装完成!")

## 2️⃣ 设置 ngrok Token

In [ ]:
# 🔑 设置你的 ngrok auth token
NGROK_AUTH_TOKEN="3AZZSmSvlfaZj8kp8quBj2MzFAY_4mtMewL9DYrkb72iQQDQv" # <-- 替换这里

if NGROK_AUTH_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("⚠️ 请先设置你的 ngrok auth token!")
    print("获取地址: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    !ngrok authtoken {NGROK_AUTH_TOKEN}
    print("✅ ngrok token 设置成功!")

## 3️⃣ 定义服务器代码（带保活机制）

In [ ]:
%%writefile colab_server.py
import os
import sys
import json
import time
import traceback
import subprocess
import gc
import threading
import signal
from flask import Flask, request, jsonify
import psutil

# 全局变量
runtime_variables = {}
start_time = time.time()
execution_lock = threading.Lock()
keep_running = True

# 创建 Flask 应用
app = Flask(__name__)

# ============== 心跳保活线程 ==============
def heartbeat_thread():
    """心跳线程，防止 Colab 休眠"""
    import requests
    last_ping = time.time()

    while keep_running:
        try:
            # 每 30 秒打印一次心跳日志
            current_time = time.strftime("%H:%M:%S")
            print(f"[心跳] {current_time} - 服务运行中", flush=True)

            # 每 5 分钟自我请求一次，保持 Colab 活跃
            if time.time() - last_ping > 300:
                try:
                    requests.get('http://localhost:5000/health', timeout=10)
                    last_ping = time.time()
                except:
                    pass

            time.sleep(30)
        except Exception as e:
            print(f"[心跳错误] {e}", flush=True)
            time.sleep(10)

# ============== API Endpoints ==============

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        "name": "ColabMCP Server",
        "version": "1.2.0",
        "status": "running",
        "uptime_minutes": round((time.time() - start_time) / 60, 2),
        "endpoints": ["/health", "/probe", "/execute", "/variables", "/files", "/cleanup"]
    })

@app.route('/health', methods=['GET'])
def health_check():
    mem = psutil.virtual_memory()
    return jsonify({
        "status": "ok",
        "uptime_minutes": round((time.time() - start_time) / 60, 2),
        "memory_available_gb": round(mem.available / (1024**3), 2),
        "memory_total_gb": round(mem.total / (1024**3), 2),
        "memory_used_pct": round(mem.percent, 2),
        "gpu_available": _check_gpu()
    })

@app.route('/probe', methods=['GET'])
def probe_environment():
    gpu_info = ""
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'],
                              capture_output=True, text=True, timeout=10)
        gpu_info = result.stdout
    except:
        gpu_info = "No GPU available"

    installed_packages = []
    try:
        result = subprocess.run(['pip', 'list', '--format=freeze'], capture_output=True, text=True, timeout=30)
        for line in result.stdout.split('\\n'):
            if '==' in line:
                installed_packages.append(line.strip())
    except:
        pass

    mem = psutil.virtual_memory()

    return jsonify({
        "gpu_info": gpu_info,
        "memory_total_gb": round(mem.total / (1024**3), 2),
        "memory_available_gb": round(mem.available / (1024**3), 2),
        "python_version": sys.version,
        "installed_packages": installed_packages[:100],
        "total_packages": len(installed_packages)
    })

@app.route('/execute', methods=['POST'])
def execute_code():
    """执行 Python 代码，带错误隔离"""
    if not execution_lock.acquire(blocking=False):
        return jsonify({
            "success": False,
            "error": "另一个代码正在执行中，请稍后重试"
        })

    try:
        data = request.get_json()
        code = data.get('code', '')
        timeout = min(data.get('timeout', 300), 600)

        if not code:
            return jsonify({"success": False, "error": "No code provided"})

        exec_globals = {'__builtins__': __builtins__, **runtime_variables}
        exec_locals = {}

        from io import StringIO
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = captured_stdout = StringIO()
        sys.stderr = captured_stderr = StringIO()

        start_exec_time = time.time()

        try:
            exec(code, exec_globals, exec_locals)

            for key, value in exec_locals.items():
                if not key.startswith('_'):
                    try:
                        json.dumps({key: str(type(value))})
                        runtime_variables[key] = value
                    except:
                        pass

            return jsonify({
                "success": True,
                "stdout": captured_stdout.getvalue(),
                "stderr": captured_stderr.getvalue(),
                "execution_time_sec": round(time.time() - start_exec_time, 3),
                "variables": list(exec_locals.keys())
            })

        except Exception as e:
            return jsonify({
                "success": False,
                "error": str(e),
                "error_type": type(e).__name__,
                "traceback": traceback.format_exc(),
                "stdout": captured_stdout.getvalue(),
                "stderr": captured_stderr.getvalue(),
                "execution_time_sec": round(time.time() - start_exec_time, 3)
            })

        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

    except Exception as e:
        return jsonify({
            "success": False,
            "error": f"服务器内部错误: {str(e)}",
            "error_type": type(e).__name__,
            "traceback": traceback.format_exc()
        })

    finally:
        execution_lock.release()

@app.route('/variables', methods=['GET'])
def list_variables():
    vars_info = {}
    for key, value in runtime_variables.items():
        try:
            var_info = {"type": str(type(value).__name__)}
            if hasattr(value, 'shape'):
                var_info["shape"] = list(value.shape) if hasattr(value.shape, '__iter__') else str(value.shape)
            if hasattr(value, '__len__'):
                try:
                    var_info["length"] = len(value)
                except:
                    pass
            vars_info[key] = var_info
        except:
            vars_info[key] = {"type": str(type(value).__name__)}

    return jsonify({"variables": vars_info, "count": len(vars_info)})

@app.route('/files', methods=['GET'])
def list_files():
    files = []
    content_dir = '/content'
    try:
        for f in os.listdir(content_dir):
            path = os.path.join(content_dir, f)
            try:
                size = os.path.getsize(path)
                files.append({
                    "name": f,
                    "path": path,
                    "size_bytes": size,
                    "size_readable": f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB",
                    "is_dir": os.path.isdir(path)
                })
            except:
                pass
    except Exception as e:
        return jsonify({"error": str(e), "files": []})
    return jsonify({"files": files, "count": len(files)})

@app.route('/cleanup', methods=['POST'])
def cleanup():
    global runtime_variables
    runtime_variables = {}
    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except:
        pass

    mem = psutil.virtual_memory()
    return jsonify({
        "success": True,
        "message": "Memory cleaned",
        "memory_available_gb": round(mem.available / (1024**3), 2)
    })

def _check_gpu():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return result.returncode == 0
    except:
        return False

def signal_handler(sig, frame):
    global keep_running
    print("\n[信号] 收到停止信号，正在关闭...")
    keep_running = False
    sys.exit(0)

if __name__ == '__main__':
    # 注册信号处理
    signal.signal(signal.SIGINT, signal_handler)
    signal.signal(signal.SIGTERM, signal_handler)

    print("\n" + "="*60)
    print("🚀 ColabMCP 服务器启动中...")
    print("="*60)
    print("版本: 1.2.0 (稳定版)")
    print("功能: 心跳保活 + 错误隔离")
    print("="*60 + "\n")

    # 启动心跳线程
    heartbeat = threading.Thread(target=heartbeat_thread, daemon=True)
    heartbeat.start()

    # 运行 Flask
    try:
        app.run(port=5000, host='0.0.0.0', threaded=True)
    except Exception as e:
        print(f"[错误] Flask 启动失败: {e}")
        raise

print("✅ 服务器代码已保存到 colab_server.py")

## 4️⃣ 启动 ngrok 隧道

In [ ]:
from pyngrok import ngrok
import time

# 清理旧的连接
try:
    ngrok.kill()
except:
    pass

time.sleep(2)

# 启动 ngrok
try:
    tunnel = ngrok.connect(5000)
    public_url = tunnel.public_url

    print("=" * 70)
    print("🎉 ngrok 隧道已建立!")
    print("=" * 70)
    print(f"\n📡 公网 URL: {public_url}")
    print(f"\n📋 复制下面的 URL 用于客户端连接:")
    print(f"   {public_url}")
    print("\n" + "-" * 70)
    print("🔧 可用端点:")
    print(f"   健康检查: {public_url}/health")
    print(f"   环境探测: {public_url}/probe")
    print(f"   执行代码: POST {public_url}/execute")
    print("-" * 70)
    print("\n⚠️ 下一个 cell 会启动 Flask 服务器，请保持运行")
    print("=" * 70)
except Exception as e:
    print(f"❌ ngrok 启动失败: {e}")
    print("\n可能的原因:")
    print("1. ngrok token 未设置或无效")
    print("2. 网络连接问题")
    raise

## 5️⃣ 启动 Flask 服务器

**重要**: 运行后请保持这个 cell 运行，不要停止！

你会看到每 30 秒打印一次心跳日志，这是正常的，表示服务正在运行。

In [ ]:
print("🚀 启动 Flask 服务器...")
print("📌 服务器正在运行，请保持这个 cell 运行")
print("📌 每 30 秒会打印心跳日志，表示服务正常")
print()

# 运行服务器
!python colab_server.py

---
## ⚠️ 防止 Colab 断开的小技巧

### 方法一：保持浏览器标签页活跃
- 不要让浏览器标签页进入后台
- 定期点击一下 Colab 页面

### 方法二：使用防休眠脚本
在浏览器控制台（F12）运行以下代码：
```javascript
function KeepAlive() {
    console.log('保持活跃...');
    document.querySelector('colab-connect-button').click();
}
setInterval(KeepAlive, 60000);
```

### 方法三：使用自动点击扩展
安装浏览器扩展，每分钟自动点击页面一次。

---
## 📝 使用说明

```python
import requests

url = "https://your-ngrok-url.ngrok-free.dev"

# 健康检查
r = requests.get(f"{url}/health")
print(r.json())
```